# RI-JK RHF Hessian：CP-HF 分解 (2) 方程求解

In [1]:
from pyscf import gto, scf, lib, df, hessian
import numpy as np
from functools import partial
from pyscf.df.grad.rhf import _int3c_wrapper

lib.num_threads(16)
np.set_printoptions(5, suppress=True, linewidth=150)
np.einsum = partial(np.einsum, optimize="greedy")

In [2]:
xyz = """
N  0   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""

mol = gto.Mole(atom=xyz, basis="def2-TZVP", max_memory=32000).build()

In [3]:
mf = scf.RHF(mol).density_fit()
mf.mo_coeff = np.load("nh3_r_hf.npz")["mo_coeff"]
mf.mo_occ = np.load("nh3_r_hf.npz")["mo_occ"]
mf.mo_energy = np.load("nh3_r_hf.npz")["mo_energy"]
mf.with_df.build()
mf.converged = True

In [4]:
mf_hess = mf.Hessian().run()
de_ref = mf_hess.de.copy()
print("de_ref shape:", de_ref.shape)

de_ref shape: (4, 4, 3, 3)


In [5]:
de_cphf = np.load("nh3_r_hf_decomp.npz")["de_cphf"]

In [6]:
mo_coeff = mf.mo_coeff
mo_occ = mf.mo_occ
mo_energy = mf.mo_energy
nao, nmo = mo_coeff.shape
mocc = mo_coeff[:, mo_occ > 0]
occ_occupation = mo_occ[mo_occ > 0]
nocc = mocc.shape[1]
dm0 = np.dot(mocc, mocc.T) * 2
dme0 = np.einsum('pi,qi,i->pq', mocc, mocc, mo_energy[mo_occ > 0]) * 2
natm = mol.natm
atmlst = range(natm)
aoslices = mol.aoslice_by_atom()
aux = mf.with_df.auxmol
auxslices = aux.aoslice_by_atom()
naux = aux.nao
mocc_2 = np.einsum("pi,i->pi", mocc, occ_occupation**0.5)
occ_energy = mo_energy[mo_occ > 0]

In [7]:
eocc = mo_energy[mo_occ > 0]
evir = mo_energy[mo_occ == 0]
mvir = mo_coeff[:, mo_occ == 0]

## Overview

In [8]:
def ovlp_deriv1_generator(mol):
    int1e_ipovlp = mol.intor("int1e_ipovlp")
    
    def get_ovlp_deriv_at_atoms(A):
        shl0, shl1, p0, p1 = aoslices[A]
        s1ao = np.zeros((3, nao, nao))
        s1ao[:, p0:p1, :] += - int1e_ipovlp[:, p0:p1] 
        s1ao[:, :, p0:p1] += - int1e_ipovlp[:, p0:p1].transpose(0, 2, 1)
        return s1ao
    return get_ovlp_deriv_at_atoms

In [9]:
f1ao = mf_hess.make_h1(mo_coeff, mo_occ)
mo1, mo_e1 = mf_hess.solve_mo1(mo_energy, mo_coeff, mo_occ, f1ao)
mo1 = np.array(mo1)
mo_e1 = np.array(mo_e1)

In [10]:
int2c2e = aux.intor("int2c2e")
int2c2e_inv = np.linalg.inv(int2c2e)
int2c2e_ip1 = aux.intor("int2c2e_ip1")
int3c2e = _int3c_wrapper(mol, aux, "int3c2e", "s1")()
int3c2e_ip1 = _int3c_wrapper(mol, aux, "int3c2e_ip1", "s1")().reshape([3, nao, nao, naux])
int3c2e_ip2 = _int3c_wrapper(mol, aux, "int3c2e_ip2", "s1")().reshape([3, nao, nao, naux])

## Response Function

### original vresp 

CP-HF/KS need response function (instead of usual fock/veff). However, for Hartree-Fock problem, response (F) is identical to potential (V).

This part need to be expanded for KS counterpart.

In [11]:
vresp = mf.gen_response()

In [12]:
dm_rand = np.random.rand(nao, nao)
dm_rand += dm_rand.T
np.allclose(vresp(dm_rand), mf.get_veff(dm=dm_rand))

True

In [27]:
np.allclose((
    + np.einsum("uvP, PQ, klQ, kl -> uv", int3c2e, int2c2e_inv, int3c2e, dm_rand)
    - 0.5 * np.einsum("uvP, PQ, klQ, vl -> uk", int3c2e, int2c2e_inv, int3c2e, dm_rand)
), vresp(dm_rand))

True

### original vind

In [15]:
def gen_vind(mf, mo_coeff, mo_occ):
    nao, nmo = mo_coeff.shape
    mocc = mo_coeff[:,mo_occ>0]
    nocc = mocc.shape[1]
    vresp = mf.gen_response(mo_coeff, mo_occ, hermi=1)
    def fx(mo1):
        mo1 = mo1.reshape(-1,nmo,nocc)
        nset = len(mo1)
        dm1 = np.empty((nset,nao,nao))
        for i, x in enumerate(mo1):
            dm = mo_coeff @ (x*2) @ mocc.T
            dm1[i] = dm + dm.T
        v1 = vresp(dm1)
        v1vo = np.empty_like(mo1)
        for i, x in enumerate(v1):
            v1vo[i] = mo_coeff.T @ x @ mocc
        return v1vo
    return fx


In [18]:
mo1_rand = np.random.rand(3, nmo, nocc)
np.allclose(gen_vind(mf, mo_coeff, mo_occ)(mo1_rand), hessian.rhf.gen_vind(mf, mo_coeff, mo_occ)(mo1_rand))

True

In [34]:
dm1_rand = mo_coeff @ (mo1_rand * 2) @ mocc.T
dm1_rand += dm1_rand.transpose(0, 2, 1)
np.allclose(
    + np.einsum("uvP, PQ, klQ, Akl, up, vi -> Api", int3c2e, int2c2e_inv, int3c2e, dm1_rand, mo_coeff, mocc)
    - 0.5 * np.einsum("uvP, PQ, klQ, Avl, up, ki -> Api", int3c2e, int2c2e_inv, int3c2e, dm1_rand, mo_coeff, mocc)
,
    hessian.rhf.gen_vind(mf, mo_coeff, mo_occ)(mo1_rand)
)

True

### modified vind

In [25]:
(
    + np.einsum("uvP, PQ, klQ, kl -> uv", int3c2e, int2c2e_inv, int3c2e, dm_rand)
    - 0.5 * np.einsum("uvP, PQ, klQ, vl -> uk", int3c2e, int2c2e_inv, int3c2e, dm_rand)
)

array([[103.48451,  65.59245,  14.02043, ...,  -1.47387,  -1.27794,   1.33156],
       [ 65.59245,  93.8889 ,  45.00369, ...,  -2.30859,  -1.95001,  -1.37275],
       [ 14.02043,  45.00369,  78.48923, ...,  -3.89764,  -3.73784, -18.83207],
       ...,
       [ -1.47387,  -2.30859,  -3.89764, ...,  76.0702 ,  -1.11735,   0.40019],
       [ -1.27794,  -1.95001,  -3.73784, ...,  -1.11735,  75.83103,   1.12707],
       [  1.33156,  -1.37275, -18.83207, ...,   0.40019,   1.12707,  78.12476]], shape=(49, 49))

In [19]:
def vresp_half_transformed(mf, mo_coeff, mo_occ):
    nao, nmo = mo_coeff.shape
    mocc = mo_coeff[:,mo_occ>0]
    nocc = mocc.shape[1]
    def fx(mo1):
        mo1_half = mo_coeff @ mo1
        resp_j  = 2 * np.einsum("uvP, PQ, klQ, vi, kj, Alj -> Auj", int3c2e, int2c2e_inv, int3c2e, mocc, mocc, mo1_half)
        resp_k  = 1 * np.einsum("uvP, PQ, klQ, vj, ki, Alj -> Auj", int3c2e, int2c2e_inv, int3c2e, mocc, mocc, mo1_half)
        resp_k += 1 * np.einsum("uvP, PQ, klQ, vj, kj, Ali -> Auj", int3c2e, int2c2e_inv, int3c2e, mocc, mocc, mo1_half)
        resp = resp_j - 0.5 * resp_k
        return mo_coeff.T @ resp
    return fx
    

In [21]:
vresp_half_transformed(mf, mo_coeff, mo_occ)(mo1_rand)[0]

array([[-2.00568, -0.37046,  1.23768,  1.32697, -0.3963 ],
       [ 0.86007, -0.72476,  0.63431,  1.00307,  0.11547],
       [ 0.96416, -0.18799, -0.42334,  0.99573,  0.15494],
       [ 0.90404, -0.03401,  0.37057, -0.45499,  0.13007],
       [ 0.92925, -0.03725,  0.74469,  0.98029, -0.51709],
       [ 0.45571, -0.21296,  0.1539 ,  0.23166, -0.00983],
       [ 0.05923, -0.0856 , -0.06364, -0.13418, -0.02441],
       [ 0.09476, -0.04314,  0.16136, -0.1677 ,  0.00314],
       [-0.04836, -0.05138,  0.07205, -0.01876, -0.11495],
       [ 0.06836, -0.13523,  0.08626,  0.02481,  0.02733],
       [-0.27568,  0.01599, -0.00328, -0.11027,  0.06433],
       [-0.58883,  0.21779, -0.18753, -0.2083 ,  0.02417],
       [ 0.29146, -0.18781,  0.0094 ,  0.10546, -0.08545],
       [ 0.26095, -0.20226, -0.04885, -0.12401, -0.1037 ],
       [ 0.11953, -0.0716 , -0.19255,  0.23436, -0.05473],
       [-0.03365,  0.06668, -0.1367 , -0.00563,  0.13251],
       [ 0.00264, -0.03176,  0.08163, -0.00895, -0.07456

In [23]:
hessian.rhf.gen_vind(mf, mo_coeff, mo_occ)(mo1_rand)[0]

array([[ 7.95792, -0.36825, -0.81115, -0.81734, -0.46466],
       [-0.36825,  5.85035, -0.61877, -0.87979, -0.34011],
       [-0.81115, -0.61877,  5.1975 , -0.16936,  0.02639],
       [-0.81734, -0.87979, -0.16936,  4.6055 , -0.15879],
       [-0.46466, -0.34011,  0.02639, -0.15879,  5.31746],
       [-0.32488,  0.5543 , -0.04195, -0.00386,  0.26313],
       [-0.47956, -0.4036 ,  0.13352,  0.33629, -0.73264],
       [-0.22194, -0.21057, -0.65196,  0.44378,  0.10361],
       [-0.39891, -0.30108, -0.3319 , -0.20035,  0.25832],
       [-0.31252,  0.21137,  0.01018, -0.31499, -0.28544],
       [-0.07647,  0.40553, -0.86142, -0.2578 , -0.78056],
       [-0.09791, -1.03644, -0.15867, -0.56571, -0.67224],
       [-0.28918,  0.00034, -0.20554, -0.13617,  0.62181],
       [-0.79257, -0.11653,  0.12728,  0.34958, -0.62882],
       [-0.30778,  0.11962,  0.47933, -0.70747, -0.13162],
       [-0.02159, -0.29717, -0.12315, -0.1129 , -0.2827 ],
       [-0.31283, -0.02142, -0.13834, -0.31239,  0.2435 

In [ ]:
hessian.rhf.gen_vind(mf, mo_coeff, mo_occ)(mo1_rand.reshape(-1, nmo, nocc)).shape

In [16]:
gen_vind(mf, mo_coeff, mo_occ)(dmo1.reshape(-1, nmo, nocc)).shape

ValueError: cannot reshape array of size 28812 into shape (49,5)

### CP-HF recover

In [24]:
def solve_withs1(
    fvind, mo_energy, mo_occ, h1, s1, max_cycle=50, tol=1e-9, level_shift=0
):
    """For field dependent basis. First order overlap matrix is non-zero.
    The first order orbitals are set to
    C^1_{ij} = -1/2 S1
    e1 = h1 - s1*e0 + (e0_j-e0_i)*c1 + vhf[c1]

    Kwargs:
        level_shift : float
            Add to diagonal terms to slightly improve the convergence speed of
            Krylov solver

    Returns:
        First order orbital coefficients (in MO basis) and first order orbital
        energy matrix
    """
    occidx = mo_occ > 0
    viridx = mo_occ == 0
    e_a = mo_energy[viridx]
    e_i = mo_energy[occidx]
    e_ai = 1 / (e_a[:, None] + level_shift - e_i)
    nvir, nocc = e_ai.shape
    nmo = nocc + nvir

    s1 = s1.reshape(-1, nmo, nocc)
    hs = mo1base = h1.reshape(-1, nmo, nocc) - s1 * e_i

    mo1base = hs.copy()
    mo1base[:, viridx] *= -e_ai
    mo1base[:, occidx] = -s1[:, occidx] * 0.5

    def vind_vo(mo1):
        mo1 = mo1.reshape(-1, nmo, nocc)
        v = fvind(mo1).reshape(-1, nmo, nocc)
        if level_shift != 0:
            v -= mo1 * level_shift
        v[:, viridx, :] *= e_ai
        v[:, occidx, :] = 0
        return v.reshape(-1, nmo * nocc)

    mo1 = lib.krylov(
        vind_vo,
        mo1base.reshape(-1, nmo * nocc),
        tol=tol,
        max_cycle=max_cycle,
    )
    mo1 = mo1.reshape(-1, nmo, nocc)
    mo1[:, occidx] = mo1base[:, occidx]

    hs += fvind(mo1).reshape(-1, nmo, nocc)
    mo1[:, viridx] = hs[:, viridx] / (e_i - e_a[:, None])

    # mo_e1 has the same symmetry as the first order Fock matrix (hermitian or
    # anti-hermitian). mo_e1 = v1mo - s1*lib.direct_sum('i+j->ij',e_i,e_i)
    mo_e1 = hs[:, occidx, :]
    mo_e1 += mo1[:, occidx] * (e_i[:, None] - e_i)

    if h1.ndim == 3:
        return mo1, mo_e1
    else:
        assert h1.ndim == 2
        return mo1[0], mo_e1[0]

In [25]:
f1ao = np.array(f1ao)
s1ao = np.array([ovlp_deriv1_generator(mol)(A) for A in range(mol.natm)])
f1mo = mo_coeff.T @ f1ao @ mocc
s1mo = mo_coeff.T @ s1ao @ mocc

In [26]:
vind = gen_vind(mf, mo_coeff, mo_occ)
t1, t2 = solve_withs1(vind, mo_energy, mo_occ, f1mo.reshape(-1, nao, nocc), s1mo.reshape(-1, nao, nocc))

In [27]:
np.allclose(mo_coeff @ t1, mo1.reshape(-1, nao, nocc))

True